# Doodle Detective — End-to-End Machine Learning Workflow

**Author:** Kaustubh  
**Goal:** Classify hand-drawn doodles of apples, cats, and stars using a Convolutional Neural Network.

This notebook walks through the complete pipeline:

1. Loading and exploring the QuickDraw dataset
2. Preprocessing images for the CNN
3. Understanding the trained CNN architecture
4. Running inference on sample drawings
5. Connecting the model to a Tkinter desktop application

---

## 1. Introduction

**What are we doing?**

We are building an interactive desktop application that recognises simple hand-drawn doodles.  
The user draws on a canvas, and a Convolutional Neural Network (CNN) predicts whether the drawing is an apple, a cat, or a star.

**Why are we doing it?**

This project demonstrates the complete machine learning workflow — from raw pixel data to a working GUI application.  
It covers dataset handling, image preprocessing, CNN design, model training, evaluation, and real-time inference.

**What should we observe?**

Pay attention to how raw sketch data is transformed into numerical tensors, how a CNN learns spatial patterns (edges, curves, shapes), and how the trained model integrates into a desktop app.

## 2. Problem Statement

**What are we doing?**

Given a 28×28 grayscale image of a hand-drawn doodle, predict one of three classes:
- Apple
- Cat
- Star

**Why is this interesting?**

Hand-drawn sketches vary enormously — different stroke widths, positions, proportions, and artistic abilities.  
A robust model must learn *invariant features*: the round shape of an apple, the pointed ears of a cat, the sharp corners of a star.

**Why 28×28?**

The QuickDraw dataset stores drawings as 28×28 grayscale bitmaps. This is small enough to train quickly on a CPU, yet large enough to preserve meaningful shape information.

## 3. Dataset Overview

**What are we doing?**

We are using the [Google QuickDraw dataset](https://quickdraw.withgoogle.com/data), a collection of 50 million hand-drawn sketches across 345 categories.

For this project we use three categories:
- Apple (~120,000 drawings)
- Cat (~120,000 drawings)
- Star (~120,000 drawings)

**Why QuickDraw?**

QuickDraw is ideal for educational CNN projects because:
- The data is freely available and well-documented
- Drawings are pre-aligned and consistently sized (28×28)
- The categories are intuitive — we can visually verify what the model learns

**Data format:**

Each drawing is stored as a 28×28 grayscale bitmap with pixel values 0–255.  
The background is black (0) and the strokes are white (255).

## 4. Dataset Exploration

**What are we doing?**

We load sample drawings from each category and visualise them to understand the data distribution.

In [ ]:
# Import libraries
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from pathlib import Path

# Add project root to path so we can import src modules
import sys, os
sys.path.insert(0, os.path.abspath('..'))

from src.config import CLASS_NAMES, INPUT_SIZE

### 4.1 Loading QuickDraw data (from .npy files)

The QuickDraw dataset is commonly distributed as NumPy `.npy` files.  
Each file contains a single array of shape `(N, 784)` where `N` is the number of drawings and each drawing is a flattened 28×28 bitmap.

If you have the dataset files, place them in the project's `data/` directory:  
- `data/full_numpy_bitmap_Apple.npy`  
- `data/full_numpy_bitmap_Cat.npy`  
- `data/full_numpy_bitmap_Star.npy`

Below is a helper to load and preview the data.

In [ ]:
def load_quickdraw_category(filepath, max_samples=10000):
    """
    Load a QuickDraw category from a .npy file.
    
    Parameters
    ----------
    filepath : str or Path
        Path to the .npy file.
    max_samples : int
        Maximum number of samples to load.
    
    Returns
    -------
    np.ndarray of shape (max_samples, 28, 28) with values in {0, 1}.
    """
    data = np.load(filepath).astype(np.float32)
    data = data[:max_samples] / 255.0  # Normalise to 0-1
    data = data.reshape(-1, 28, 28)     # Reshape to 2D images
    return data


def visualize_samples(data_dict, samples_per_class=5):
    """Display a grid of sample drawings from each class."""
    fig, axes = plt.subplots(
        len(data_dict), samples_per_class,
        figsize=(samples_per_class * 2, len(data_dict) * 2)
    )
    if len(data_dict) == 1:
        axes = [axes]
    
    for row, (class_name, images) in enumerate(data_dict.items()):
        indices = np.random.choice(len(images), samples_per_class, replace=False)
        for col, idx in enumerate(indices):
            ax = axes[row][col] if len(data_dict) > 1 else axes[col]
            ax.imshow(images[idx], cmap="gray", vmin=0, vmax=1)
            ax.axis("off")
            if col == 0:
                ax.set_ylabel(class_name, fontsize=12, fontweight="bold")
    
    plt.suptitle("Sample QuickDraw Drawings", fontsize=14, fontweight="bold")
    plt.tight_layout()
    plt.show()


# Uncomment and run if you have the data files:
# data_dir = Path('../data')
# data = {}
# for name in CLASS_NAMES:
#     path = data_dir / f'full_numpy_bitmap_{name}.npy'
#     if path.exists():
#         data[name] = load_quickdraw_category(path)
#         print(f'{name}: {len(data[name])} samples loaded')
#
# if data:
#     visualize_samples(data)

**What to observe:**
- Drawings vary in stroke thickness, position, and style.
- Some are clear, others are abstract — the model must handle both.
- The data is binary (black background, white strokes), which simplifies preprocessing.

## 5. Image Preprocessing

**What are we doing?**

Before feeding drawings into the CNN, we must convert them into a consistent numerical format.

**The preprocessing pipeline (from `src/preprocess.py`):**

1. **Resize** to 28×28 using LANCZOS resampling (antialiased downscaling).
2. **Normalise** pixel values from [0, 255] to [0.0, 1.0].
3. **Invert** the colours — the QuickDraw dataset has *white strokes on black*, but our drawing canvas produces *black strokes on white*. Inversion fixes this mismatch.
4. **Reshape** to `(1, 28, 28, 1)` — the 4D tensor shape that Keras expects for batch inference.

**Why these steps?**
- Resizing ensures all inputs have the same dimensions.
- Normalisation helps the neural network converge faster (large pixel values can destabilise training).
- Inversion aligns our canvas with the training data distribution.

In [ ]:
# Demonstrate the preprocessing pipeline
from PIL import Image, ImageDraw

# Create a synthetic doodle (black cross on white background)
demo_image = Image.new("L", (280, 280), color=255)
demo_draw = ImageDraw.Draw(demo_image)
demo_draw.line([(50, 50), (230, 230)], fill=0, width=20)
demo_draw.line([(230, 50), (50, 230)], fill=0, width=20)

# Visualise before and after preprocessing
from src.preprocess import preprocess_for_model

tensor = preprocess_for_model(demo_image)
print(f"Input shape:  {demo_image.size}")
print(f"Output shape: {tensor.shape}")
print(f"Value range:  [{tensor.min():.3f}, {tensor.max():.3f}]")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(8, 4))
ax1.imshow(demo_image, cmap="gray", vmin=0, vmax=255)
ax1.set_title("Original (280×280)")
ax1.axis("off")

ax2.imshow(tensor[0, :, :, 0], cmap="gray", vmin=0, vmax=1)
ax2.set_title("Preprocessed (28×28, inverted)")
ax2.axis("off")
plt.tight_layout()
plt.show()

**What to observe:**
- The 280×280 drawing is downscaled to 28×28. Fine details are lost, but the overall shape is preserved.
- The colours are inverted: black lines become white on black background.
- The output tensor shape is `(1, 28, 28, 1)` — ready for `model.predict()`.

## 6. CNN Architecture

**What are we doing?**

We load the pre-trained model and inspect its architecture to understand the layers that process the drawings.

**Why this architecture?**

The network follows a standard CNN design pattern for small images:
1. **Convolutional layers** extract visual features (edges, corners, textures)
2. **Pooling layers** reduce spatial dimensions, making the network focus on *what* is present rather than *where* it is
3. **Dense layers** perform high-level classification based on the extracted features

This is a classic educational CNN — not over-engineered, but effective for the task.

In [ ]:
# Load the trained model
from src.predict import load_model

model = load_model()
model.summary()

**What to observe:**
- Total parameter count — is the model small enough to train quickly?
- How the spatial dimensions shrink from 28×28 → 14×14 → 7×7 as we go through Conv + Pool layers.
- The final Dense layer has 3 neurons (one per class) with a softmax activation — this produces a probability distribution over Apple, Cat, Star.

### 6.1 Visualising the architecture

Here's a text diagram of the network flow:

In [ ]:
tf.keras.utils.plot_model(model, show_shapes=True, show_layer_names=False, dpi=60)

## 7. Training Process

**What are we doing?**

The model was trained on ~300,000 QuickDraw samples (100,000 per class) using:
- **Optimiser:** Adam (adaptive learning rate)
- **Loss function:** Sparse Categorical Crossentropy (standard for multi-class classification)
- **Batch size:** 32 or 64
- **Epochs:** 10–20 (with early stopping to prevent overfitting)
- **Train/Validation split:** 80/20

**Why these choices?**
- Adam is the default choice for CNNs — it works well without extensive tuning.
- Crossentropy loss measures how well the predicted probabilities match the true labels.
- Early stopping prevents the model from memorising the training data.

The training code (for reference) follows this pattern:

In [ ]:
# Pseudocode / reference for the training loop:
#
# from tensorflow.keras import layers, models
#
# model = models.Sequential([
#     layers.Conv2D(32, (3, 3), activation='relu', input_shape=(28, 28, 1)),
#     layers.MaxPooling2D((2, 2)),
#     layers.Conv2D(64, (3, 3), activation='relu'),
#     layers.MaxPooling2D((2, 2)),
#     layers.Flatten(),
#     layers.Dense(128, activation='relu'),
#     layers.Dense(3, activation='softmax')
# ])
#
# model.compile(
#     optimizer='adam',
#     loss='sparse_categorical_crossentropy',
#     metrics=['accuracy']
# )
#
# history = model.fit(
#     x_train, y_train,
#     validation_data=(x_val, y_val),
#     epochs=15,
#     batch_size=64
# )

print("See Doodle CNN Model.ipynb for the original training implementation.")

## 8. Model Evaluation

**What are we doing?**

We evaluate the trained model on a held-out test set to measure its real-world performance.

In [ ]:
# Evaluate on test data (requires test set to be loaded)
# test_loss, test_acc = model.evaluate(x_test, y_test, verbose=0)
# print(f"Test accuracy: {test_acc:.3f}")
# print(f"Test loss:     {test_loss:.3f}")

print("To evaluate, load a held-out test set and call model.evaluate().")

### 8.1 Confusion Matrix

A confusion matrix shows which classes the model confuses most often.

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns

# y_pred = np.argmax(model.predict(x_test), axis=1)
# cm = confusion_matrix(y_test, y_pred)
#
# plt.figure(figsize=(6, 5))
# sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
#             xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
# plt.xlabel('Predicted')
# plt.ylabel('True')
# plt.title('Confusion Matrix')
# plt.show()
#
# print(classification_report(y_test, y_pred, target_names=CLASS_NAMES))

print("Run this cell after loading x_test and y_test to see the confusion matrix.")

**What to look for:**
- Which class has the highest accuracy?
- Are there systematic confusions (e.g., apples mistaken for stars)?
- Does the model fail on certain drawing styles?

## 9. Saving the Model

**What are we doing?**

After training, we save the model in the `.keras` format (Keras v3 format, introduced in TensorFlow 2.6+).

**Why this format?**
- Single file containing both architecture and weights.
- Can be loaded with `tf.keras.models.load_model()`.
- No dependency on the original Python code — the model is self-contained.

In [ ]:
# Save the model
# model.save('../models/doodle_classifier.keras')
# print("Model saved to models/doodle_classifier.keras")

from src.config import MODEL_PATH
print(f"The trained model is stored at: {MODEL_PATH}")

## 10. Running Inference

**What are we doing?**

We test the model on individual drawings — both from the dataset and from synthetic inputs — to see how it behaves.

In [ ]:
from src.preprocess import preprocess_for_model
from src.predict import predict_doodle

# Create a synthetic "star" doodle
star = Image.new("L", (280, 280), color=255)
d = ImageDraw.Draw(star)
# A simple 5-point star
points = [(140, 20), (170, 110), (260, 110), (190, 170),
          (215, 260), (140, 205), (65, 260), (90, 170),
          (20, 110), (110, 110)]
d.polygon(points, fill=0)

# Preprocess and predict
tensor = preprocess_for_model(star)
result = predict_doodle(model, tensor)

print(f"Predicted:    {result['class_name']}")
print(f"Confidence:   {result['confidence']:.1f}%")
print(f"Probabilities:")
for name, prob in zip(CLASS_NAMES, result['probabilities']):
    print(f"  {name}: {prob:.1%}")

fig, ax = plt.subplots(figsize=(3, 3))
ax.imshow(tensor[0, :, :, 0], cmap="gray", vmin=0, vmax=1)
ax.set_title(f"Prediction: {result['class_name']}")
ax.axis("off")
plt.show()

**What to observe:**
- The model should predict "Star" with high confidence for this synthetic star.
- The confidence scores sum to 1.0 — a property of the softmax output layer.
- Try modifying the drawing to see how the prediction changes.

## 11. Connecting the Model to the Tkinter GUI

**What are we doing?**

The trained model is loaded into a desktop application built with Tkinter (Python's standard GUI library).

**How it works:**

1. The user draws on a 280×280 canvas using the mouse.
2. Each stroke is mirrored onto a PIL `Image` object (grayscale, white background).
3. When the mouse is released, the image is:
   - Preprocessed via `preprocess_for_model()` (resize → normalise → invert → reshape)
   - Passed to `predict_doodle()`
4. The results update the GUI: prediction label, confidence percentage, bar chart, fun fact, and stats.

**Why Tkinter?**

Tkinter is included with Python — no additional GUI framework installation required. It keeps the project self-contained and easy to run.

In [ ]:
# Launch the GUI from within the notebook (if running locally)
# import tkinter as tk
# from src.gui import DoodleDetectiveApp
#
# root = tk.Tk()
# app = DoodleDetectiveApp(root)
# root.mainloop()

print("To launch the GUI, run:  python run_gui.py")
print("Or uncomment the code above and run this cell.")

### 11.1 Data flow diagram

```
User draws on canvas
         │
         ▼
PIL Image (280×280, grayscale)
         │
         ▼
preprocess.py ──► resize to 28×28
                 ► normalise to [0, 1]
                 ► invert colours
                 ► reshape to (1, 28, 28, 1)
         │
         ▼
predict.py ──► model.predict(tensor)
              ► argmax → predicted class
              ► softmax → confidence %
         │
         ▼
GUI updates:
  - Prediction label
  - Confidence label
  - Bar chart (visualization.py)
  - Fun fact (random choice)
  - Drawing stats
```

## 12. Conclusion

**What we accomplished:**

- Built a CNN that classifies hand-drawn doodles into three categories.
- Trained it on the Google QuickDraw dataset.
- Deployed it in a desktop application with a drawing canvas and live predictions.

**Key takeaways:**

1. **CNNs learn spatial hierarchies** — early layers detect edges, later layers combine them into shapes.
2. **Preprocessing matters** — normalisation and colour inversion are critical for matching the training data distribution.
3. **Small models can work well** — a network with ~1 million parameters is sufficient for simple shape classification.
4. **End-to-end ML** — the pipeline from dataset to GUI is the same pattern used in production systems.

**What next?**

This project can be extended in many directions (see the Future Improvements section in the README).

---

*Thank you for reading!*